<a href="https://colab.research.google.com/github/yassinemaataoui/Colab_project/blob/main/Analyse_ventes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq && apt-get install -y openjdk-11-jdk
version = "3.5.6"
hadoop_version = "hadoop3"
url = f"https://downloads.apache.org/spark/spark-{version}/spark-{version}-bin-{hadoop_version}.tgz"
!wget {url} -O spark-{version}-bin-{hadoop_version}.tgz
!ls -lh spark-{version}-bin-{hadoop_version}.tgz
!tar -xvzf spark-{version}-bin-{hadoop_version}.tgz
!pip install -q findspark
import os, findspark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/spark-{version}-bin-{hadoop_version}"
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Atelier_PySpark").getOrCreate()
spark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk is already the newest version (11.0.28+6-1ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
--2025-11-04 22:42:55--  https://downloads.apache.org/spark/spark-3.5.6/spark-3.5.6-bin-hadoop3.tgz
Resolving downloads.apache.org (downloads.apache.org)... 135.181.214.104, 88.99.208.237, 2a01:4f8:10a:39da::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|135.181.214.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400923510 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.6-bin-hadoop3.tgz’

spark-3.5.6-bin-had 100%[===================>] 382.35M  11.9MB/s    in 32s     

2025-11-04 22:43:29 (11.8 MB/s) - ‘spark-3.5.6-bin-

# Compte Rendu : Analyse des ventes avec Spark SQL
**I. Objectif**

L'objectif de cet exercice est de manipuler et analyser les données des ventes contenues dans un fichier CSV ventes.csv en utilisant Spark SQL, afin de calculer des agrégations, filtrer des informations, créer de nouvelles vues et identifier les tendances des ventes

**Introduction**

Dans le cadre de cet exercice pratique, nous allons explorer l'utilisation de Spark SQL pour analyser un dataset de ventes contenu dans un fichier CSV nommé ventes.csv. Ce dataset comprend plusieurs colonnes essentielles telles que la date de la vente, la région géographique, le produit vendu, la quantité achetée et le prix unitaire. L'objectif principal est de démontrer comment combiner la puissance de Spark avec la simplicité du langage SQL pour effectuer des manipulations de données, des agrégations et des analyses avancées. Spark SQL permet de traiter des volumes massifs de données de manière distribuée, ce qui est particulièrement utile dans un environnement Big Data. Tout au long de ce compte rendu, nous allons suivre les étapes prescrites dans le cahier d'exercices, en fournissant non seulement le code nécessaire, mais aussi un explication détaillée pour chaque étape. Chaque section sera présentée sous forme de paragraphes élaborés, expliquant le contexte, la logique derrière les opérations, et les implications des résultats obtenus. Cela permettra une compréhension approfondie des concepts clés tels que les vues temporaires, les requêtes d'agrégation, les fonctions SQL avancées, et les jointures. En fin de compte, cette analyse vise à extraire des insights commerciaux précieux, tels que les régions les plus performantes, les produits les plus populaires, et les tendances temporelles des ventes. Pour mener à bien cette analyse, nous utiliserons PySpark, qui est l'interface Python de Spark, en créant une session Spark et en chargeant les données dans un DataFrame. Assurez-vous que le fichier ventes.csv est disponible dans votre répertoire de travail, et que Spark est correctement installé et configuré dans votre environnement.

**Mise en place de l’environnement**

Pour commencer, il faut créer une session Spark (SparkSession) qui permettra de lire les fichiers CSV et d’exécuter les requêtes SQL. Ensuite, on crée le fichier CSV local contenant les données de ventes, avec les colonnes : date, région, produit, quantité et prix. Le fichier est chargé dans un DataFrame Spark, et on vérifie la structure et le contenu avec show() et printSchema().

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
with open("ventes.csv", "w") as f:

SyntaxError: incomplete input (ipython-input-2440160869.py, line 2)

**Étape 1 : Créer une Vue Temporaire ventes_view**

In [ ]:
df.createOrReplaceTempView("ventes_view")

+ Cette vue permet d'accéder aux données via des requêtes SQL standard. Elle est temporaire et accessible uniquement dans cette session Spark.
+ Maintenant, toutes les requêtes suivantes utiliseront ventes_view au lieu du DataFrame directement.

Pour faciliter l'exécution des requêtes SQL tout au long de l'analyse, nous créons une vue temporaire à partir du DataFrame principal. Cette vue, nommée ventes_view, agit comme une table virtuelle qui référence les données du DataFrame sans les dupliquer physiquement. En utilisant df.createOrReplaceTempView("ventes_view"), nous rendons les données accessibles via des requêtes SQL standard, ce qui combine la flexibilité de SQL avec la puissance de Spark. Les vues temporaires sont particulièrement utiles dans les environnements distribués car elles permettent de réutiliser des ensembles de données complexes sans avoir à recharger les fichiers à chaque fois. Contrairement aux vues globales, cette vue est limitée à la session Spark actuelle, assurant ainsi une isolation et une sécurité des données. Une fois créée, toutes les requêtes suivantes pourront utiliser ventes_view comme une table SQL classique, permettant des opérations telles que les sélections, les filtres et les agrégations. Cette approche est essentielle pour une analyse itérative, où nous pouvons construire des requêtes complexes étape par étape. Par exemple, si le dataset était plus volumineux, cette vue permettrait d'optimiser les performances en évitant les recalculs inutiles. En résumé, cette étape transforme notre DataFrame en une ressource SQL prête à l'emploi, ouvrant la voie à des analyses plus sophistiquées.

**Étape 2 : Nombre Total d'Unités Vendues par Région (Trié Décroissant)**

Nous calculons la somme des quantités par région et trions les résultats par ordre décroissant.

In [ ]:
df.columns

[]

In [ ]:
result2 = spark.sql("""
    SELECT region, SUM(quantite) AS total_unites
    FROM ventes_view
    GROUP BY region
    ORDER BY total_unites DESC
""")
result2.show()

AnalysisException: [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column or function parameter with name `region` cannot be resolved. ; line 2 pos 11;
'Sort ['total_unites DESC NULLS LAST], true
+- 'Aggregate ['region], ['region, 'SUM('quantite) AS total_unites#25]
   +- SubqueryAlias ventes_view
      +- View (`ventes_view`, [])
         +- Relation [] csv


SUM(quantite) agrège les quantités pour chaque région.
GROUP BY region regroupe les données par région.
ORDER BY total_unites DESC trie les résultats du plus élevé au plus bas.
Cela nous aide à identifier les régions avec le plus de ventes en volume.



Dans cette étape, nous nous intéressons à l'agrégation des ventes par région pour identifier les zones géographiques les plus actives en termes de volume. La requête SQL SELECT region, SUM(quantite) AS total_unites FROM ventes_view GROUP BY region ORDER BY total_unites DESC calcule la somme totale des quantités vendues pour chaque région, puis trie les résultats par ordre décroissant. Cette analyse est fondamentale pour comprendre la répartition géographique des ventes, permettant aux entreprises de cibler leurs efforts marketing ou logistiques sur les régions les plus performantes. Par exemple, une région avec un total élevé d'unités pourrait indiquer une demande forte, justifiant des investissements supplémentaires en stock ou en promotion. Le tri décroissant met en évidence les leaders, facilitant la prise de décisions stratégiques. En utilisant Spark SQL, cette opération est exécutée de manière distribuée, ce qui signifie que même avec des millions de lignes, le calcul reste efficace. Les résultats peuvent révéler des disparités régionales, potentiellement dues à des facteurs démographiques, économiques ou saisonniers. De plus, cette métrique peut être combinée avec d'autres analyses, comme les revenus, pour une vue plus complète. En pratique, cette information aide à optimiser la chaîne d'approvisionnement, en évitant les surstocks dans les régions moins actives et en renforçant la présence dans les zones à forte demande.

**Étape 3 : Prix Maximum par Produit et Produit le Plus Cher**
Nous affichons le prix maximum pour chaque produit et identifions le produit global le plus cher.

In [ ]:
result3 = spark.sql("""
    SELECT produit, MAX(prix) AS prix_max
    FROM ventes_view
    GROUP BY produit
""")
result3.show()

# Pour identifier le produit le plus cher globalement
max_prix_global = spark.sql("""
    SELECT produit, prix
    FROM ventes_view
    ORDER BY prix DESC
    LIMIT 1
""")
max_prix_global.show()

Ici, nous explorons la variabilité des prix au sein des produits pour identifier les valeurs extrêmes et les articles les plus coûteux. La première requête, SELECT produit, MAX(prix) AS prix_max FROM ventes_view GROUP BY produit, agrège les prix maximums par produit, offrant un aperçu de la fourchette de prix pour chaque catégorie. Cela est utile pour analyser la stratégie de tarification, en détectant si certains produits ont des prix fluctuants ou stables. Ensuite, pour identifier le produit le plus cher globalement, nous utilisons SELECT produit, prix FROM ventes_view ORDER BY prix DESC LIMIT 1, qui trie toutes les transactions par prix décroissant et sélectionne la première. Cette approche révèle non seulement le produit le plus onéreux, mais aussi son prix exact, ce qui peut influencer les décisions d'achat ou de promotion. Dans un contexte commercial, connaître les prix maximums aide à évaluer la valeur perçue des produits et à ajuster les stratégies de vente. Par exemple, un produit avec un prix maximum élevé pourrait être positionné comme un article premium, attirant une clientèle spécifique. De plus, cette analyse peut mettre en lumière des anomalies, comme des erreurs de saisie de données, nécessitant une validation supplémentaire. En combinant ces requêtes, nous obtenons une vue d'ensemble de la structure des prix, essentielle pour l'optimisation des marges bénéficiaires et la segmentation du marché

**Étape 4 : Colonne Calculée total_vente et Chiffre d'Affaires par Produit**
Nous ajoutons une colonne calculée total_vente = quantite * prix et calculons la somme des ventes totales par produit.

In [ ]:
result4 = spark.sql("""
    SELECT produit, SUM(quantite * prix) AS total_vente
    FROM ventes_view
    GROUP BY produit
""")
result4.show()

Cette étape introduit le calcul du chiffre d'affaires en créant une colonne dérivée total_vente, qui représente le revenu généré par chaque transaction (quantité multipliée par prix). La requête SELECT produit, SUM(quantite * prix) AS total_vente FROM ventes_view GROUP BY produit agrège ces valeurs par produit, fournissant une mesure directe de la performance financière de chaque article. Cette métrique est cruciale pour évaluer la contribution de chaque produit aux revenus totaux, permettant d'identifier les best-sellers et les articles moins rentables. Par exemple, un produit avec un chiffre d'affaires élevé pourrait justifier des investissements en marketing ou en production, tandis qu'un produit à faible revenu pourrait être candidat à une réduction de stock ou à une révision de prix. En utilisant des calculs dans les requêtes SQL, nous évitons de modifier le dataset original, préservant l'intégrité des données. Cette approche est particulièrement puissante dans Spark, où les opérations sont optimisées pour les gros volumes. De plus, cette analyse peut être étendue à des comparaisons temporelles ou régionales, offrant des insights plus profonds. En résumé, le calcul du chiffre d'affaires transforme des données brutes en indicateurs clés de performance, essentiels pour la prise de décisions stratégiques dans le domaine des ventes.

**Étape 5 : Analyse Mensuelle des Ventes pour 2025**
Nous extrayons l'année et le mois de la colonne date, puis affichons le chiffre d'affaires total par mois pour l'année 2025.

In [ ]:
result5 = spark.sql("""
    SELECT
        SUBSTRING(date, 1, 4) AS annee,
        SUBSTRING(date, 6, 2) AS mois,
        SUM(quantite * prix) AS total_vente_mensuel
    FROM ventes_view
    WHERE SUBSTRING(date, 1, 4) = '2025'
    GROUP BY annee, mois
    ORDER BY mois
""")
result5.show()

Pour analyser les tendances temporelles, nous extrayons l'année et le mois de la colonne date et calculons le chiffre d'affaires mensuel pour l'année 2025. La requête utilise SUBSTRING(date, 1, 4) pour isoler l'année et SUBSTRING(date, 6, 2) pour le mois, puis agrège les ventes avec SUM(quantite * prix). En filtrant sur l'année 2025 avec WHERE, nous nous concentrons sur une période spécifique, et le tri par mois permet de visualiser l'évolution saisonnière. Cette analyse est indispensable pour détecter des patterns, tels que des pics de ventes pendant certaines périodes (par exemple, les fêtes de fin d'année). Elle aide les entreprises à planifier leurs inventaires, leurs campagnes promotionnelles et leurs ressources humaines en fonction des fluctuations mensuelles. Par exemple, si les ventes augmentent en octobre, cela pourrait indiquer une saisonnalité liée à des événements spécifiques. En utilisant des fonctions de chaîne dans SQL, nous manipulons les dates sans nécessiter de bibliothèques externes, ce qui est efficace dans un environnement Spark. De plus, cette approche peut être étendue à des analyses annuelles ou hebdomadaires pour une granularité plus fine. En fin de compte, comprendre les tendances temporelles permet d'anticiper les demandes futures et d'optimiser les opérations commerciales.

**Étape 6 : Ventes où le Chiffre d'Affaires Total Dépasse 10 000**
Nous affichons toutes les ventes individuelles où quantite * prix > 10000.

In [ ]:
result6 = spark.sql("""
    SELECT *
    FROM ventes_view
    WHERE quantite * prix > 10000
""")
result6.show()

Dans cette étape, nous filtrons les transactions individuelles où le revenu dépasse un seuil de 10 000 unités monétaires, en utilisant SELECT * FROM ventes_view WHERE quantite * prix > 10000. Cette requête identifie les ventes à haut valeur, qui pourraient représenter des commandes importantes ou des clients premium. Analyser ces transactions est essentiel pour comprendre les sources de revenus élevés, potentiellement liées à des produits coûteux ou à des volumes importants. Par exemple, une vente dépassant ce seuil pourrait indiquer une opportunité de fidélisation client ou de cross-selling. Cette information aide également à détecter des anomalies, comme des erreurs de données, et à évaluer l'impact des ventes exceptionnelles sur les performances globales. En filtrant directement dans la requête, nous évitons de traiter inutilement des données moins pertinentes, optimisant ainsi les performances. De plus, cette analyse peut être combinée avec des dimensions temporelles ou régionales pour une vue plus nuancée. En résumé, identifier les ventes à haut revenu permet de prioriser les efforts sur les segments les plus lucratifs de l'activité commerciale.

**Étape 7 : Produit le Plus Vendu en Quantité par Région**

Pour chaque région, nous identifions le produit avec la quantité totale la plus élevée.

In [ ]:
result7 = spark.sql("""
    SELECT region, produit, SUM(quantite) AS total_quantite
    FROM ventes_view
    GROUP BY region, produit
    ORDER BY region, total_quantite DESC
""")
# Pour simplifier, nous pouvons utiliser une sous-requête ou ROW_NUMBER pour le top par région
result7.show()

Pour chaque région, nous identifions le produit dominant en termes de quantité vendue en agrégeant SUM(quantite) par région et produit, puis en triant par région et quantité décroissante. Cette requête révèle les préférences régionales, aidant à adapter les stratégies de distribution et de marketing. Par exemple, si un produit spécifique domine dans une région, cela pourrait refléter des goûts locaux ou des besoins spécifiques. Cette analyse est cruciale pour l'optimisation des stocks, en évitant les ruptures dans les régions où un produit est populaire. Bien que la requête affiche tous les résultats, elle peut être raffinée avec des fonctions de fenêtre pour sélectionner uniquement le top par région. En utilisant Spark SQL, cette opération gère efficacement les données distribuées, permettant des analyses scalables. De plus, combiner cette métrique avec les revenus offre une vue complète de la performance. En définitive, comprendre les produits leaders par région soutient des décisions éclairées pour maximiser les ventes et la satisfaction client.

**Étape 8 : Prix Minimum et Maximum par Produit**

Nous calculons le prix min et max pour chaque produit.

In [ ]:
result8 = spark.sql("""
    SELECT produit, MIN(prix) AS prix_min, MAX(prix) AS prix_max
    FROM ventes_view
    GROUP BY produit
""")
result8.show()

Nous calculons les prix extrêmes pour chaque produit avec SELECT produit, MIN(prix) AS prix_min, MAX(prix) AS prix_max FROM ventes_view GROUP BY produit. Cette analyse met en lumière la variabilité des prix, utile pour évaluer la stabilité des tarifs et détecter des opportunités de remises ou d'ajustements. Par exemple, une grande différence entre min et max pourrait indiquer des promotions ou des variations saisonnières. Cette information est essentielle pour la gestion des marges et la stratégie de tarification. En agrégeant par produit, nous obtenons un résumé concis, facilitant les comparaisons. Spark optimise ces calculs pour de gros datasets, assurant une exécution rapide. En résumé, analyser les prix extrêmes aide à affiner les politiques commerciales et à améliorer la compétitivité.

**Étape 9 : Créer une Vue ventes_analyse**

Nous créons une nouvelle vue avec les colonnes spécifiées, incluant total_vente, annee, et mois.

In [ ]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW ventes_analyse AS
    SELECT
        region,
        produit,
        quantite,
        prix,
        quantite * prix AS total_vente,
        SUBSTRING(date, 1, 4) AS annee,
        SUBSTRING(date, 6, 2) AS mois
    FROM ventes_view
""")

Nous créons une vue temporaire ventes_analyse avec des colonnes calculées comme total_vente, annee et mois, en utilisant CREATE OR REPLACE TEMP VIEW. Cette vue pré-calcule les données nécessaires, améliorant l'efficacité des analyses suivantes. Elle inclut toutes les colonnes demandées, servant de base pour les requêtes avancées. Cette approche réduit les répétitions et optimise les performances dans Spark. En résumé, cette vue centralise les transformations, facilitant une analyse itérative et cohérente.

**Étape 10 : Top 3 Produits par Chiffre d'Affaires**

Nous affichons les 3 produits avec le plus haut SUM(total_vente), triés décroissant.

In [ ]:
result10 = spark.sql("""
    SELECT produit, SUM(total_vente) AS total_ca
    FROM ventes_analyse
    GROUP BY produit
    ORDER BY total_ca DESC
    LIMIT 3
""")
result10.show()

Nous identifions les trois produits les plus rentables avec SELECT produit, SUM(total_vente) AS total_ca FROM ventes_analyse GROUP BY produit ORDER BY total_ca DESC LIMIT 3. Cette analyse met en évidence les best-sellers financiers, guidant les priorités en marketing et inventaire. Par exemple, investir dans ces produits peut booster les revenus. Le tri et la limitation assurent une sélection précise. Spark gère efficacement cette agrégation. En résumé, cette étape révèle les piliers du chiffre d'affaires.

**Étape 11 : Chiffre d'Affaires Total par Région et Région la Plus Performante**

Nous calculons le total par région et identifions la région avec le plus haut chiffre d'affaires.

In [ ]:
result11 = spark.sql("""
    SELECT region, SUM(total_vente) AS total_ca_region
    FROM ventes_analyse
    GROUP BY region
    ORDER BY total_ca_region DESC
""")
result11.show()

# Région avec le plus haut CA
top_region = spark.sql("""
    SELECT region, SUM(total_vente) AS total_ca_region
    FROM ventes_analyse
    GROUP BY region
    ORDER BY total_ca_region DESC
    LIMIT 1
""")
top_region.show()

Enfin, nous calculons le total des ventes par région et identifions la leader avec des requêtes d'agrégation et de tri. Cette analyse compare les performances régionales, aidant à allouer les ressources. Par exemple, la région top pourrait bénéficier d'expansions. Les résultats guident les stratégies globales. En utilisant Spark, nous traitons les données à grande échelle. En résumé, cette étape clôt l'analyse en fournissant des insights stratégiques.

**Conclusion**

Ce notebook a démontré comment utiliser Spark SQL pour analyser un dataset de ventes. Nous avons couvert la création de vues, les requêtes de base, les agrégations, les jointures implicites via les vues, et les fonctions avancées. Les résultats peuvent être utilisés pour prendre des décisions commerciales, comme cibler les régions à fort potentiel ou optimiser les stocks de produits populaires. Pour des analyses plus poussées, nous pourrions intégrer des visualisations ou des modèles de machine learning.